# Bolão Copa do Mundo FIFA 2026 — Previsão de placares com Rede Neural (TensorFlow/Keras)

**Aluno:** Bruno Aires &nbsp;|&nbsp; **Turma:** 2º BIM 2026

Pipeline reprodutível que treina uma **rede neural de regressão de Poisson** para prever os
placares exatos dos **24 jogos da 1ª rodada da fase de grupos** da Copa do Mundo 2026 e exporta
o resultado em JSON no formato exigido por `bolao_copa.txt`.

## 1. Introdução

**Hipótese do modelo.** O número de gols marcados por cada seleção em uma partida é uma
**contagem discreta não-negativa** que pode ser modelada por uma distribuição de **Poisson**,
cujo parâmetro de intensidade $\lambda$ (gols esperados) é uma **função não-linear** da força
relativa entre as equipes. Essa força é capturada por: rating **Elo** (construído
cronologicamente sobre 20 anos de jogos), **ranking/pontos FIFA**, **forma recente**
(gols pró/contra e aproveitamento nos últimos jogos) e a **probabilidade implícita do mercado de
apostas** (odds 1/X/2 convertidas). A vantagem de mando de campo **não é ajustada manualmente**:
fornecemos apenas a *feature* `neutral` (e o status de país-sede) e deixamos a rede aprender o efeito.

**Data Leakage.** Para evitar que a rede neural sofra de vazamento de dados ao mapear
o Elo diretamente como proxy perfeito das odds no treino, as probabilidades históricas implícitas
foram calibradas adicionando ruído via **Distribuição de Dirichlet**. Isso mimetiza a incerteza e o
spread real das casas de apostas no passado, permitindo que o modelo generalize corretamente ao
receber as cotações reais trazidas de `Copa_do_Mundo_Cotacoes.pdf` para os jogos de 2026.

**Abordagem.** Uma rede densa recebe o vetor de atributos do confronto e produz **dois valores**
$(\lambda_A, \lambda_B)$ — gols esperados de cada seleção — via ativação `softplus` (garante
$\lambda>0$). O treinamento minimiza a **log-verossimilhança negativa de Poisson**. Cada partida
histórica é ponderada por **decay temporal** (ênfase nos últimos 2 anos). Para seleções com
histórico esparso aplicamos **transfer learning** por confederação (encolhimento bayesiano
das estatísticas em direção à média da confederação). A previsão final de cada jogo é um
**placar inteiro determinístico**: a **moda** da Poisson de cada $\lambda$, restrita ao intervalo
histórico realista $[0,5]$. **Todas as previsões derivam exclusivamente da rede neural**.

### Configuração e reprodutibilidade

Fixamos *seeds* em `PYTHONHASHSEED`, `random`, `numpy` e `tensorflow`, e habilitamos operações
determinísticas no TF. Assim, execuções subsequentes do notebook produzem **exatamente os mesmos
resultados**.

In [19]:
# =============================================================================
# CÉLULA 1 — Imports, Seeds e Configuração de Reprodutibilidade
# =============================================================================
# OBJETIVO: Importar todas as bibliotecas necessárias e garantir que o pipeline
# seja 100% reprodutível — qualquer re-execução do notebook produz os mesmos
# resultados numéricos, bit a bit.
#
# A pasta data e seus arquivos disponiveis no github devem ser carregadas neste notebook para a correta execução.
#
# REPRODUTIBILIDADE
# Redes neurais envolvem inicialização aleatória de pesos, dropout estocástico
# e operações paralelas no hardware. Sem fixar seeds, cada execução produziria
# previsões ligeiramente diferentes — inaceitável para um bolão onde o placar
# exato deve ser determinístico e auditável.
# =============================================================================

import os, random, json, io, warnings

# Semente global única: todas as fontes de aleatoriedade derivam deste valor.
# O número 42 é uma convenção popular em ciência de dados (sem relevância matemática).
SEED = 42

# PYTHONHASHSEED: controla o hash de strings/objetos Python, que afeta dicts e sets
# internamente usados pelo TensorFlow e Pandas.
os.environ["PYTHONHASHSEED"] = str(SEED)

# TF_DETERMINISTIC_OPS: força o TensorFlow a usar apenas operações determinísticas.
# Algumas operações GPU (ex.: scatter, segment_sum) têm resultados não-determinísticos
# por padrão — esta flag as desabilita em favor da reprodutibilidade.
os.environ["TF_DETERMINISTIC_OPS"] = "1"

# TF_CUDNN_DETERMINISTIC: aplica o mesmo controle às operações cuDNN (convoluções, etc.).
# Irrelevante em CPU pura, mas necessário para rodar em GPU sem surpresas.
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

# Suprime logs verbosos do TensorFlow (nível 2 = apenas erros críticos).
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# Suprime warnings do Python (ex.: FutureWarning do Pandas) para saída limpa.
warnings.filterwarnings("ignore")

# Imports principais:
# - numpy/pandas: manipulação de dados tabulares e vetores numéricos
# - tensorflow/keras: construção, treinamento e inferência da rede neural
# - scipy.stats.poisson: cálculo da PMF (função de massa de probabilidade) de Poisson
#   para converter lambdas contínuos em placares inteiros via moda
# - scipy.stats.dirichlet: geração de perturbação probabilística nas odds históricas
#   (técnica de calibração para evitar data leakage — ver Seção 4)
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from scipy.stats import poisson, dirichlet

# Fixamos seeds nas três camadas de aleatoriedade do Python:
# 1. random (módulo nativo Python): shuffles, amostras, etc.
# 2. numpy: geração de arrays aleatórios, inicialização de pesos em alguns backends
# 3. keras/tf: pesos iniciais da rede, dropout, batch shuffling
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Habilita determinismo em nível de operador TF (disponível a partir do TF 2.8+).
# O bloco try/except captura ambientes mais antigos onde a função não existe.
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print("determinism aviso:", e)

# Confirmação de versões: importante para diagnóstico de incompatibilidades futuras.
print("TensorFlow:", tf.__version__, "| NumPy:", np.__version__, "| Pandas:", pd.__version__)

TensorFlow: 2.20.0 | NumPy: 2.0.2 | Pandas: 2.2.2


## 2. Calendário e dados dos jogos

Coletamos o calendário oficial da 1ª rodada da fase de grupos (datas, horários, sedes, grupos e
pareamentos). O notebook recorre à tabela embutida corrigida conforme chaves oficiais da FIFA de 2026.

In [20]:
# =============================================================================
# CÉLULA 2 — Calendário Oficial da 1ª Rodada (Fase de Grupos)
# =============================================================================
# OBJETIVO: Definir os 24 confrontos da 1ª rodada da Copa do Mundo 2026 com
# todas as metainformações necessárias para o modelo: siglas das seleções,
# data, horário, grupo e se a seleção A é país-sede.
#
# ESTRATÉGIA DE FALLBACK:
# O código tenta buscar o calendário atualizado da Wikipedia. Se não houver
# acesso à internet (ex.: ambiente de produção restrito), usa a tabela FIXTURES
# embutida diretamente no código — garantindo que o pipeline funcione offline.
#
# SOBRE O CAMPO A_sede:
# Indica se a Seleção A é um dos três países-sede (EUA, México ou Canadá).
# Quando True, neutral=0 no vetor de features (não é campo neutro — há vantagem
# de sede). A rede neural aprende sozinha o peso desse efeito a partir dos dados.
# =============================================================================

# Tabela de fixtures embutida: fonte de verdade offline.
# Formato: (número do jogo, sigla_mandante, sigla_visitante, data, hora, grupo, é_sede)
# As siglas seguem o padrão FIFA de 3 letras (ex.: BRA, ESP, GER).
FIXTURES = [
    # jogo, sigla_A, sigla_B, data,        hora,    grupo, A_é_sede
    (1,  "MEX","RSA","2026-06-11","16:00","A", True ),
    (2,  "KOR","CZE","2026-06-11","23:00","A", False),
    (3,  "CAN","BIH","2026-06-12","16:00","B", True ),
    (4,  "USA","PAR","2026-06-12","22:00","D", True ),
    (5,  "HAI","SCO","2026-06-13","22:00","C", False),
    (6,  "AUS","TUR","2026-06-14","01:00","D", False),
    (7,  "BRA","MAR","2026-06-13","19:00","C", False),
    (8,  "QAT","SUI","2026-06-13","16:00","B", False),
    (9,  "CIV","ECU","2026-06-14","13:00","E", False),
    (10, "GER","CUW","2026-06-14","14:00","E", False),
    (11, "NED","JPN","2026-06-14","17:00","F", False),
    (12, "SWE","TUN","2026-06-14","23:00","F", False),
    (13, "KSA","URU","2026-06-15","19:00","H", False),
    (14, "ESP","CPV","2026-06-15","13:00","H", False),
    (15, "IRN","NZL","2026-06-15","22:00","G", False),
    (16, "BEL","EGY","2026-06-15","16:00","G", False),
    (17, "FRA","SEN","2026-06-16","16:00","I", False),
    (18, "IRQ","NOR","2026-06-16","19:00","I", False),
    (19, "ARG","ALG","2026-06-16","22:00","J", False),
    (20, "AUT","JOR","2026-06-17","01:00","J", False),
    (21, "GHA","PAN","2026-06-17","20:00","L", False),
    (22, "ENG","CRO","2026-06-17","17:00","L", False),
    (23, "POR","COD","2026-06-17","14:00","K", False),
    (24, "UZB","COL","2026-06-17","23:00","K", False),
]

import requests

def fetch_calendar():
    """
    Tenta buscar o HTML da página de grupos da Copa 2026 na Wikipedia.
    Retorna o texto HTML se bem-sucedido, ou None se offline/timeout.
    O timeout de 8s evita que o notebook trave em ambientes sem internet.
    """
    try:
        url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_group_stage"
        r = requests.get(url, timeout=8, headers={"User-Agent":"Mozilla/5.0"})
        r.raise_for_status()  # lança exceção se status HTTP >= 400
        return r.text
    except Exception as e:
        print("[calendario] fetch online indisponivel -> usando tabela embutida.")
        return None

# Tentativa de fetch online (resultado ignorado — apenas para log/debug futuro).
# O calendário efetivamente usado é sempre o FIXTURES embutido, já validado.
_html = fetch_calendar()

# Converte a lista de tuplas em DataFrame Pandas para facilitar operações vetoriais.
# A coluna 'confronto' é uma string legível usada em logs e saídas intermediárias.
cal = pd.DataFrame(FIXTURES, columns=["jogo","A","B","data","hora","grupo","A_sede"])
cal["confronto"] = cal["A"] + " x " + cal["B"]
print(f"{len(cal)} jogos da 1a rodada carregados.")

[calendario] fetch online indisponivel -> usando tabela embutida.
24 jogos da 1a rodada carregados.


## 3. Fontes de dados e Odds

In [21]:
# =============================================================================
# CÉLULA 3 — Odds do Mercado de Apostas e Conversão em Probabilidades Implícitas
# =============================================================================
# OBJETIVO: Incorporar o sinal do mercado de apostas como feature do modelo.
# As odds representam a estimativa coletiva de milhares de apostadores e analistas,
# capturando informações que rankings e histórico de resultados podem não refletir
# (ex.: lesões recentes, estado psicológico de equipes, condições climáticas).
#
# FONTE DAS ODDS:
# Extraídas manualmente do arquivo Copa_do_Mundo_Cotacoes.pdf para os 24 jogos da casa de Apostas Betano em 30/05/2026.
# Formato: (odd_vitória_A, odd_empate, odd_vitória_B) — escala europeia (decimal).
# Ex.: odd 1.09 para ESP×CPV indica que o mercado vê a Espanha como enorme favorita.
#
# CONVERSÃO PARA PROBABILIDADES IMPLÍCITAS:
# P(resultado) = (1/odd) / soma(1/odds)
# A normalização pela soma remove o "vig" (margem da casa) e converte as odds em
# probabilidades que somam 1.0 — essas são as features imp_1, imp_x e imp_2.
#
# INTERPRETAÇÃO PRÁTICA:
# - Jogo 10 (GER×CUW): odds 1.04 / 14.5 / 55.0 → mercado dá ~93% de chance de
#   vitória alemã — informação que o Elo captura de forma mais suavizada.
# - Jogo 18 (IRQ×NOR): odds 11.0 / 5.9 / 1.26 → Noruega é grande favorita.
# =============================================================================

# Dicionário: chave = número do jogo, valor = tupla (odd_1, odd_X, odd_2)
ODDS = {
    1:(1.50,4.25,6.90),  2:(2.75,3.10,2.75),  3:(1.83,3.65,4.40),  4:(2.05,3.35,3.75),
    5:(6.60,4.35,1.50),  6:(4.40,3.80,1.78),  7:(1.62,3.80,5.80),  8:(10.75,5.60,1.28),
    9:(3.85,2.85,2.30),  10:(1.04,14.50,55.00), 11:(1.98,3.75,3.60), 12:(1.95,3.35,4.15),
    13:(6.50,4.35,1.50), 14:(1.09,10.00,30.00), 15:(1.93,3.45,4.15), 16:(1.67,3.85,5.10),
    17:(1.47,4.40,6.80), 18:(11.00,5.90,1.26), 19:(1.42,4.50,8.25), 20:(1.37,5.20,8.00),
    21:(1.98,3.55,3.80), 22:(1.78,3.70,4.55), 23:(1.28,5.70,10.75), 24:(7.90,4.50,1.42),
}

def odds_to_prob(o1, ox, o2):
    """
    Converte odds decimais (europeias) em probabilidades implícitas normalizadas.
    1/odd = probabilidade bruta; divisão pela soma remove a margem da casa (over-round).
    Retorna array [P(vitória_A), P(empate), P(vitória_B)] que somam 1.0.
    """
    inv = np.array([1.0/o1, 1.0/ox, 1.0/o2])  # probabilidades brutas
    return inv / inv.sum()                       # normalização para remover o vig

# Calcula probabilidades implícitas para cada jogo e as armazena num dicionário.
imp = {j: odds_to_prob(*o) for j, o in ODDS.items()}

# Adiciona as três probabilidades implícitas como colunas do DataFrame de fixtures.
# np.vstack empilha os arrays numa matriz (24, 3) que o Pandas expande em 3 colunas.
cal[["imp_1","imp_x","imp_2"]] = np.vstack([imp[int(j)] for j in cal["jogo"]])

In [22]:
# =============================================================================
# CÉLULA 4 — Carregamento dos Dados Históricos de Resultados e Ranking FIFA
# =============================================================================
# OBJETIVO: Carregar as duas fontes de dados históricas que formam a base de
# treinamento do modelo:
#
# 1. results.csv (repositório martj42/international_results no GitHub):
#    Contém todos os jogos internacionais de seleções desde 1872. Cada linha
#    traz: data, times, placar, torneio, cidade, país e se foi campo neutro.
#    São ~49 mil partidas — cobertura histórica excepcional.
#
# 2. fifa_ranking.csv (arquivo local):
#    Séries históricas do ranking FIFA mensais (~62 mil entradas).
#    Usamos para extrair ranking e pontos FIFA na data mais próxima de cada
#    jogo histórico, via merge temporal (merge_asof).
#
# ESTRATÉGIA DE FALLBACK (igual à célula do calendário):
# Tenta download online; se falhar, carrega o CSV local em DATA_DIR='data/'.
# =============================================================================

DATA_DIR = "data"  # diretório local onde os CSVs ficam armazenados como backup

# URL pública do repositório de resultados internacionais (atualizado frequentemente).
# Usar a URL do raw.githubusercontent.com garante acesso direto ao CSV sem parsing HTML.
RESULTS_URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

def load_results():
    """
    Baixa os resultados históricos de seleções direto do GitHub.
    parse_dates=["date"] converte a coluna de data para datetime64 automaticamente,
    habilitando operações temporais (filtros, merge_asof, DateOffset) mais adiante.
    """
    try:
        resp = requests.get(RESULTS_URL, timeout=15)  # timeout generoso para arquivo grande
        resp.raise_for_status()
        # io.StringIO converte o texto da resposta HTTP em objeto file-like para o Pandas
        df = pd.read_csv(io.StringIO(resp.text), parse_dates=["date"])
        print("[results] baixado online:", df.shape)
    except Exception as e:
        # Fallback: lê o CSV local pré-baixado
        df = pd.read_csv(os.path.join(DATA_DIR, "results.csv"), parse_dates=["date"])
        print("[results] local:", df.shape)
    return df

# Carrega os ~49 mil resultados históricos
raw = load_results()

# Ranking FIFA: arquivo local (não disponível como URL pública gratuita).
# rank_date é a data de publicação do ranking; usada no merge_asof para
# buscar o ranking vigente na data de cada partida histórica.
fifa = pd.read_csv(os.path.join(DATA_DIR, "fifa_ranking.csv"), parse_dates=["rank_date"])
print("[fifa_ranking]:", fifa.shape)

[results] baixado online: (49437, 9)
[fifa_ranking]: (62424, 9)


## 4. Pré-processamento e Modelagem de Odds Históricas

In [24]:
# =============================================================================
# CÉLULA 5 — Padronização de Nomes e Mapeamento de Abreviações
# =============================================================================
# OBJETIVO: Garantir que os nomes de países no dataset de ranking FIFA e no
# dataset de resultados históricos sejam idênticos, permitindo joins corretos.
# Discrepâncias de nomenclatura (ex.: "Korea Republic" vs "South Korea") causariam
# NaN silenciosos que degradariam as features — um bug difícil de detectar.
#
#
# 1. FIFA_RENAME: corrige nomes não-padrão do ranking FIFA para o padrão
#    usado no dataset de resultados históricos.
#
# 2. ABRV2NAME: mapeia siglas FIFA de 3 letras para nomes completos em inglês.
#    Necessário para unir os fixtures (que usam siglas) ao histórico (nomes completos).
#    Ex.: "BRA" → "Brazil", "COD" → "DR Congo".
#
# 3. name2conf: dicionário nome_país → confederação (UEFA, CONMEBOL, CAF, etc.).
#    Usado depois para o encolhimento bayesiano por confederação (Seção 6).
# =============================================================================

# Mapeamento de correção de nomes: chave=nome_errado, valor=nome_correto
# Importante manter atualizado caso o dataset FIFA mude a nomenclatura.
FIFA_RENAME = {
    "Korea Republic":"South Korea",
    "USA":"United States",
    "Côte d'Ivoire":"Ivory Coast",
    "Cabo Verde":"Cape Verde",
    "IR Iran":"Iran",
    "Congo DR":"DR Congo",
    "China PR":"China",
    "Kyrgyz Republic":"Kyrgyzstan",
    "Cape Verde Islands":"Cape Verde"
}
fifa["country_full"] = fifa["country_full"].replace(FIFA_RENAME)

# Mapeamento sigla → nome completo para as 48 seleções participantes da Copa 2026.
# Inclui todas as seleções dos 24 jogos da 1ª rodada.
ABRV2NAME = {
    "MEX":"Mexico","RSA":"South Africa","KOR":"South Korea","CZE":"Czech Republic",
    "CAN":"Canada","BIH":"Bosnia and Herzegovina","USA":"United States","PAR":"Paraguay",
    "HAI":"Haiti","SCO":"Scotland","AUS":"Australia","TUR":"Turkey","BRA":"Brazil","MAR":"Morocco",
    "QAT":"Qatar","SUI":"Switzerland","CIV":"Ivory Coast","ECU":"Ecuador","GER":"Germany",
    "CUW":"Curaçao","NED":"Netherlands","JPN":"Japan","SWE":"Sweden","TUN":"Tunisia",
    "KSA":"Saudi Arabia","URU":"Uruguay","ESP":"Spain","CPV":"Cape Verde","IRN":"Iran",
    "NZL":"New Zealand","BEL":"Belgium","EGY":"Egypt","FRA":"France","SEN":"Senegal","IRQ":"Iraq",
    "NOR":"Norway","ARG":"Argentina","ALG":"Algeria","AUT":"Austria","JOR":"Jordan","GHA":"Ghana",
    "PAN":"Panama","ENG":"England","CRO":"Croatia","POR":"Portugal","COD":"DR Congo",
    "UZB":"Uzbekistan","COL":"Colombia"
}

# Extrai a confederação de cada país a partir da entrada mais recente do ranking FIFA.
# drop_duplicates(keep='last') após sort garante o registro mais atualizado por país.
# Resultado: dicionário nome → confederação (ex.: "Brazil" → "CONMEBOL").
name2conf = (fifa.sort_values("rank_date")
                 .drop_duplicates("country_full", keep="last")
                 .set_index("country_full")["confederation"]
                 .to_dict())
def conf_of(name): return name2conf.get(name, "OTHER")  # fallback para países sem ranking

# Pré-processamento básico dos resultados históricos:
# - Remove partidas sem placar (dados faltantes ou adiados)
# - Converte placares para inteiro (podem vir como float após dropna)
# - Ordena cronologicamente: CRÍTICO para o cálculo incremental do Elo (Célula 6)
df = raw.dropna(subset=["home_score","away_score"]).copy()
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)
df = df.sort_values("date").reset_index(drop=True)

# Salva a data do jogo mais recente no histórico.
# Usada depois como referência para: decay temporal, janela de treino e janela de validação.
DATA_MAX = df["date"].max()

In [25]:
# =============================================================================
# CÉLULA 6 — Cálculo Incremental de Elo e Features de Forma Recente
# =============================================================================
# OBJETIVO: Construir, jogo a jogo na ordem cronológica, duas features dinâmicas
# que capturam a força e o momento atual de cada seleção:
#
# A) RATING ELO
# Sistema de rating adaptado do xadrez (Arpad Elo, 1960). Muito usado em esportes
# e e-sports. A lógica central:
#   - Cada time começa em 1500 pontos
#   - Após cada jogo, pontos são transferidos do perdedor para o vencedor
#   - O delta depende do resultado inesperado: vencer um favorito vale mais pontos
#   - O fator K (intensidade da atualização) é amplificado pela margem de gols
#     (log1p(|gols_H - gols_A|) + 1), valorizando vitórias convincentes
#
# B) FORMA RECENTE (deque de K_FORM=6 jogos)
# Para cada seleção, mantemos um buffer circular dos últimos 6 jogos com:
#   - gols marcados (gf), gols sofridos (ga), pontos ganhos (ppg)
# O deque (collections.deque com maxlen) automaticamente descarta o jogo mais
# antigo quando um novo é adicionado — estrutura perfeita para janelas deslizantes.
#
# IMPORTANTE: As features de Elo e forma registradas para cada jogo histórico
# são os valores ANTES de processar aquele jogo (sem look-ahead = sem leakage).
# =============================================================================

from collections import deque, defaultdict

# Hiperparâmetros do sistema Elo:
K_FORM = 6        # número de jogos recentes para calcular a forma
ELO_BASE = 1500.0 # rating inicial de toda seleção (convenção padrão)
ELO_K = 30.0      # fator K base: controla a velocidade de atualização do rating

# defaultdict com lambda: qualquer seleção que apareça pela primeira vez recebe
# automaticamente o rating base 1500, sem precisar inicializar explicitamente.
elo  = defaultdict(lambda: ELO_BASE)
form = defaultdict(lambda: deque(maxlen=K_FORM))  # buffer circular por seleção
cnt  = defaultdict(int)  # contador de jogos históricos por seleção

def form_stats(t):
    """
    Retorna as médias de (gols_marcados, gols_sofridos, pontos_por_jogo)
    dos últimos K_FORM jogos da seleção t.
    Retorna (NaN, NaN, NaN) se a seleção ainda não tem jogos registrados —
    esses NaNs serão tratados na Célula 7 (preenchimento por média da confederação).
    """
    dq = form[t]
    if not dq: return (np.nan, np.nan, np.nan)
    a = np.asarray(dq, float)
    return (a[:,0].mean(), a[:,1].mean(), a[:,2].mean())

# Dicionário de listas para armazenar eficientemente as features de cada jogo.
# Usar dicionário de listas + conversão final para colunas do DataFrame é muito
# mais rápido que crescer o DataFrame linha a linha (evita re-alocações de memória).
cols = {k: [] for k in ["elo_h","elo_a","f_gf_h","f_ga_h","f_ppg_h","f_gf_a","f_ga_a","f_ppg_a"]}

# Loop principal: percorre TODOS os ~49 mil jogos históricos em ordem cronológica.
# Para cada jogo: (1) registra features pré-jogo, depois (2) atualiza Elo e forma.
for h, a, hs, asc in zip(df["home_team"], df["away_team"], df["home_score"], df["away_score"]):

    # === PASSO 1: Registra features ANTES do jogo (sem look-ahead) ===
    eh, ea = elo[h], elo[a]          # ratings atuais
    gfh, gah, pph = form_stats(h)    # forma recente do mandante
    gfa, gaa, ppa = form_stats(a)    # forma recente do visitante

    cols["elo_h"].append(eh); cols["elo_a"].append(ea)
    cols["f_gf_h"].append(gfh); cols["f_ga_h"].append(gah); cols["f_ppg_h"].append(pph)
    cols["f_gf_a"].append(gfa); cols["f_ga_a"].append(gaa); cols["f_ppg_a"].append(ppa)

    # === PASSO 2: Atualiza o Elo com o resultado real do jogo ===
    # Probabilidade esperada de vitória do mandante segundo o Elo atual
    exp_h = 1.0 / (1.0 + 10 ** (-(eh - ea) / 400.0))  # fórmula Elo clássica
    res_h = 1.0 if hs > asc else (0.5 if hs == asc else 0.0)  # resultado real: 1/0.5/0

    # Delta amplificado pela margem de gols: vitória por 3x0 vale mais que 1x0.
    # log1p(|diff|) + 1 evita delta=0 e cresce sub-linearmente (diminishing returns).
    delta = ELO_K * (np.log1p(abs(hs - asc)) + 1.0) * (res_h - exp_h)
    elo[h] = eh + delta  # vencedor ganha pontos
    elo[a] = ea - delta  # perdedor perde a mesma quantidade (jogo de soma zero)

    # === PASSO 3: Atualiza a forma recente (buffer circular) ===
    # Pontos da tabela: 3 para vitória, 1 para empate, 0 para derrota
    ph = 3 if hs > asc else (1 if hs == asc else 0)  # pontos do mandante
    pa = 3 if asc > hs else (1 if hs == asc else 0)   # pontos do visitante
    # Formato do registro: (gols_marcados, gols_sofridos, pontos)
    form[h].append((hs, asc, ph))  # perspectiva do mandante
    form[a].append((asc, hs, pa))  # perspectiva do visitante (gols invertidos)
    cnt[h] += 1; cnt[a] += 1

# Adiciona as features calculadas como colunas do DataFrame histórico.
for k, v in cols.items():
    df[k] = v

# Salva o estado final do Elo (após o último jogo histórico).
# Esses valores serão usados na inferência dos 24 jogos da Copa 2026.
ELO_FINAL = dict(elo)

In [26]:
# =============================================================================
# CÉLULA 7 — Join Temporal com Ranking FIFA, Calibração de Odds e Janela de Treino
# =============================================================================
# OBJETIVO: (A) Enriquecer o dataset histórico com ranking/pontos FIFA na data
# de cada jogo via merge temporal; (B) Gerar probabilidades implícitas históricas
# calibradas com perturbação Dirichlet (solução ao data leakage); (C) Definir
# o vetor de features final (FEATS) e filtrar a janela de 20 anos para treino.
#
# (A) MERGE TEMPORAL (merge_asof):
# Para cada jogo histórico, queremos o ranking FIFA vigente na época — não o atual.
# pd.merge_asof(direction='backward') faz exatamente isso: para cada data de jogo,
# busca o ranking publicado mais recentemente antes ou igual àquela data.
#
# (B) SOLUÇÃO AO DATA LEAKAGE — Calibração com Dirichlet:
# Problema: nos dados históricos, NÃO temos odds reais das casas de apostas.
# Opção ingênua: usar probabilidades Elo como proxy das odds. Isso cria leakage
# porque Elo e odds-derivadas-do-Elo seriam quase perfeitamente correlacionadas,
# fazendo o modelo aprender a relação trivialmente — mas ela quebraria na inferência
# real (onde as odds têm informação além do Elo).
# Solução: perturbamos as probabilidades Elo com ruído Dirichlet, simulando a
# variabilidade real das odds de mercado. O parâmetro alpha=base*50 garante que
# a perturbação seja realista (não excessiva, mas suficiente para quebrar a
# correlação perfeita).
# =============================================================================

# --- (A) Join temporal com ranking FIFA ---

# Prepara DataFrame de ranking para o merge: renomeia colunas para evitar conflito
# com os nomes existentes no DataFrame de jogos.
rk = (fifa[["rank_date","country_full","rank","total_points"]]
      .rename(columns={"rank_date":"date"})
      .sort_values("date"))  # OBRIGATÓRIO ordenar antes do merge_asof

DEF_RANK, DEF_PTS = 150.0, 0.0  # valores padrão para países sem histórico no ranking FIFA

# Garante tipo str para evitar erros de join por tipo misto
df["home_team"] = df["home_team"].astype(str)
df["away_team"] = df["away_team"].astype(str)
rk["country_full"] = rk["country_full"].astype(str)

# merge_asof para mandante e visitante separadamente:
# - on='date': coluna de alinhamento temporal (ambos devem estar ordenados por ela)
# - by='team': garante que o merge respeite a identidade do time
# - direction='backward': pega o ranking publicado mais recentemente antes do jogo
for side in ["home","away"]:
    r2 = rk.rename(columns={
        "country_full": f"{side}_team",
        "rank": f"rank_{side[0]}",
        "total_points": f"pts_{side[0]}"
    })
    df = pd.merge_asof(df, r2, on="date", by=f"{side}_team", direction="backward")

# Preenche NaN com defaults para países nunca ranqueados pela FIFA
df["rank_h"] = df["rank_h"].fillna(DEF_RANK); df["rank_a"] = df["rank_a"].fillna(DEF_RANK)
df["pts_h"]  = df["pts_h"].fillna(DEF_PTS);   df["pts_a"]  = df["pts_a"].fillna(DEF_PTS)
df = df.reset_index(drop=True)

# --- (B) Calibração das odds históricas com perturbação Dirichlet ---

def elo_to_1x2_calibrated(eh, ea, seed=SEED):
    """
    Gera probabilidades 1/X/2 históricas calibradas com perturbação Dirichlet.

    Passo 1: Converte diferença de Elo em probabilidade de vitória do mandante.
    Passo 2: Estima P(empate) como função da igualdade entre os times.
      - P(empate) é máxima quando os times são iguais (Elo diff ≈ 0)
      - Decresce conforme a diferença aumenta (clássicos são mais equilibrados)
    Passo 3: Adiciona ruído Dirichlet com concentração alpha = base_prob * 50.
      - Alpha alto (>10) → distribuição Dirichlet concentrada perto das probabilidades base
      - Simula que o mercado acerta o 'centro' mas adiciona spread realista
    """
    np.random.seed(seed)
    ph = 1.0 / (1.0 + 10 ** (-(eh - ea) / 400.0))  # P(vitória mandante) pelo Elo
    pdraw = 0.26 * (1.0 - np.abs(2 * ph - 1.0))    # P(empate): 26% máx. em jogos equilibrados
    p1 = ph * (1 - pdraw)   # P(vitória A) ajustada
    p2 = (1 - ph) * (1 - pdraw)  # P(vitória B) ajustada
    base_probs = np.vstack([p1, pdraw, p2]).T  # matriz (N_jogos, 3)

    # Adiciona ruído via Dirichlet para simular variabilidade real do mercado de apostas
    calibrated = []
    for bp in base_probs:
        alpha = bp * 50.0  # concentração: quanto maior, menos ruído
        calibrated.append(dirichlet.rvs(alpha)[0])  # amostra uma distribuição de prob.
    res = np.array(calibrated)
    return res[:,0], res[:,1], res[:,2]  # retorna três arrays: P(1), P(X), P(2)

# Aplica a calibração a todos os jogos históricos
p1, px, p2 = elo_to_1x2_calibrated(df["elo_h"].values, df["elo_a"].values)
df["imp_1"], df["imp_x"], df["imp_2"] = p1, px, p2

# --- (C) Vetor de features e janela temporal de treino ---

# Feature auxiliar: diferença direta de Elo entre mandante e visitante.
# Redundante com elo_h e elo_a, mas ajuda a rede a capturar o efeito relativo
# mais diretamente, sem precisar subtrai-los ela mesma.
df["elo_diff"] = df["elo_h"] - df["elo_a"]
df["neutral"]  = df["neutral"].astype(int)  # bool → int (0=casa, 1=neutro)

# Filtro temporal: apenas os últimos 20 anos de jogos entram no treinamento.
# Motivação: jogos de 50 anos atrás têm pouca relevância preditiva para o futebol
# moderno (regras, condicionamento físico e estilo de jogo mudaram drasticamente).
WINDOW_START = DATA_MAX - pd.DateOffset(years=20)
train_df = df[df["date"] >= WINDOW_START].copy()

# Vetor de features: ordem importa (deve ser idêntica na inferência dos jogos de 2026)
FEATS = [
    "elo_h", "elo_a", "elo_diff",           # força absoluta e relativa via Elo
    "rank_h", "rank_a", "pts_h", "pts_a",   # ranking e pontos FIFA
    "f_gf_h", "f_ga_h", "f_ppg_h",          # forma recente do mandante
    "f_gf_a", "f_ga_a", "f_ppg_a",          # forma recente do visitante
    "neutral",                               # mando de campo
    "imp_1", "imp_x", "imp_2"               # probabilidades implícitas do mercado
]

# Preenche eventuais NaN restantes com a média da coluna.
# Isso ocorre principalmente nas features de forma dos primeiros jogos de uma seleção.
for c in FEATS:
    if train_df[c].isna().any():
        train_df[c] = train_df[c].fillna(train_df[c].mean())

print("Janela de treino configurada:", len(train_df), "amostras.")

Janela de treino configurada: 19332 amostras.


In [27]:
# =============================================================================
# CÉLULA 8 — Decay Temporal, Matriz de Features e Normalização
# =============================================================================
# OBJETIVO: Preparar X (features), Y (alvos) e W (pesos) para o treinamento.
#
# DECAY TEMPORAL (sample weights):
# Jogos mais antigos têm menor relevância para prever o futebol atual.
# Modelamos isso com uma função de decaimento exponencial de meia-vida 2 anos:
#   w(t) = 0.5 ^ (idade_em_anos / 2)
#   → jogo de hoje: peso 1.0
#   → jogo de 2 anos atrás: peso 0.5
#   → jogo de 4 anos atrás: peso 0.25
# O TensorFlow aceita sample_weight no .fit() e aplica esses pesos na função
# de perda — efetivamente dando mais atenção ao futebol recente.
#
# NORMALIZAÇÃO (StandardScaler):
# Redes neurais são muito sensíveis à escala das features. Features com escala
# grande (ex.: Elo ~1500-2000, pontos FIFA ~500-1800) dominariam features pequenas
# (ex.: probabilidades 0-1) no cálculo do gradiente, dificultando o treinamento.
# StandardScaler subtrai a média e divide pelo desvio padrão: z = (x - μ) / σ.
# IMPORTANTE: o scaler é ajustado APENAS nos dados de treino e depois aplicado
# ao conjunto de validação e inferência — sem olhar os dados futuros.
# =============================================================================

# Decay exponencial: calcula a idade de cada jogo em anos a partir do último jogo histórico.
HALF_LIFE_Y = 2.0  # meia-vida em anos — hiperparâmetro ajustável
age_y = (DATA_MAX - train_df["date"]).dt.days / 365.25  # idade em anos fracionários
train_df["w"] = np.power(0.5, age_y / HALF_LIFE_Y)  # w(t) = 0.5^(t/T_half)

from sklearn.preprocessing import StandardScaler

# Extrai arrays NumPy — mais eficiente que DataFrame para TensorFlow.
# float32 é o dtype nativo do TensorFlow (float64 é desnecessariamente pesado).
X = train_df[FEATS].values.astype("float32")                    # shape: (N_jogos, 17)
Y = train_df[["home_score","away_score"]].values.astype("float32")  # shape: (N_jogos, 2)
W = train_df["w"].values.astype("float32")                      # shape: (N_jogos,)

# Ajusta o scaler nos dados de treino e transforma imediatamente.
# O objeto scaler é salvo para transformar os jogos de 2026 com os mesmos parâmetros.
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X).astype("float32")  # features normalizadas: shape (N_jogos, 17)

## 5. Modelo TensorFlow / Keras

In [28]:
# =============================================================================
# CÉLULA 9 — Arquitetura da Rede Neural (Poisson Score Net)
# =============================================================================
# OBJETIVO: Definir e compilar a rede neural densa que mapeia o vetor de 17
# features para os dois lambdas da distribuição de Poisson (λ_A, λ_B).
#
# ARQUITETURA: Rede densa decrescente 17 → 128 → 64 → 32 → 2
# Total: 13.474 parâmetros — intencionalmente pequena para evitar overfitting
# em ~19 mil amostras de treinamento.
#
# DECISÕES DE DESIGN:
#
# • Ativação ReLU nas camadas intermediárias:
#   Padrão eficiente para dados tabulares. Evita o problema do vanishing gradient
#   (comum com sigmoid/tanh em redes profundas).
#
# • BatchNormalization após cada Dense:
#   Normaliza as ativações de cada mini-batch, acelerando a convergência e
#   reduzindo a sensibilidade à taxa de aprendizado inicial.
#
# • Dropout (30% e 20%):
#   Regularização estocástica: desativa aleatoriamente neurônios durante o
#   treinamento, forçando a rede a aprender representações redundantes.
#   Reduz overfitting. Dropout maior na primeira camada (mais parâmetros → maior risco).
#
# • Ativação Softplus na camada de saída (λ_A, λ_B):
#   softplus(x) = log(1 + e^x) ≈ max(0, x) mas suavizado.
#   Garante que λ > 0 em todos os casos (Poisson exige λ positivo).
#   Vantagem sobre ReLU: é diferenciável em x=0 (gradiente nunca é exatamente 0).
#
# • Perda: keras.losses.Poisson()
#   Implementa a log-verossimilhança negativa de Poisson:
#   L(λ, y) = λ - y*log(λ) + log(y!)
#   Onde y são os gols reais e λ são os gols esperados previstos pela rede.
#   Minimizar esta perda é equivalente a maximizar a probabilidade de Poisson
#   de observar os placares reais — fundamento estatístico sólido.
#
# • Otimizador: Adam (lr=1e-3)
#   Adaptativo por parâmetro, robusto para dados tabulares e gradientes esparsos.
# =============================================================================

def build_model(n_features, seed=SEED):
    """
    Constrói e compila a rede neural de regressão de Poisson.

    Args:
        n_features: número de features de entrada (17 neste pipeline)
        seed: semente para inicialização determinística dos pesos

    Returns:
        Modelo Keras compilado, pronto para treinamento.
    """
    tf.keras.utils.set_random_seed(seed)
    # GlorotUniform (Xavier): inicializa pesos com variância proporcional à
    # soma das dimensões de entrada e saída — ajuda o gradiente a fluir
    # uniformemente pelas camadas no início do treinamento.
    init = keras.initializers.GlorotUniform(seed=seed)

    # Camada de entrada: aceita vetores de tamanho n_features (17)
    inp = keras.Input(shape=(n_features,), name="features")

    # Bloco 1: 128 neurônios + BN + Dropout 30%
    x = keras.layers.Dense(128, activation="relu", kernel_initializer=init)(inp)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.30, seed=seed)(x)  # desativa 30% dos neurônios no treino

    # Bloco 2: 64 neurônios + BN + Dropout 20%
    x = keras.layers.Dense(64, activation="relu", kernel_initializer=init)(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.20, seed=seed)(x)

    # Bloco 3: 32 neurônios sem dropout (camada de representação final)
    x = keras.layers.Dense(32, activation="relu", kernel_initializer=init)(x)

    # Camada de saída: 2 neurônios com softplus → (λ_A, λ_B), sempre positivos
    lam = keras.layers.Dense(2, activation="softplus", kernel_initializer=init, name="lambdas")(x)

    model = keras.Model(inp, lam, name="poisson_score_net")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),  # taxa de aprendizado inicial = 0.001
        loss=keras.losses.Poisson()              # NLL de Poisson como função de perda
    )
    return model

model = build_model(len(FEATS))
model.summary()  # exibe a tabela de camadas, shapes e parâmetros

Model: "poisson_score_net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 17)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │         2,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambdas (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,474 (52.63 KB)

 Trainable params: 13,090 (51.13 KB)

 Non-trainable params: 384 (1.50 KB)

In [29]:
# =============================================================================
# CÉLULA 10 — Treinamento da Rede Neural com Early Stopping e ReduceLROnPlateau
# =============================================================================
# OBJETIVO: Treinar o modelo nos dados históricos com validação cronológica e
# parada automática quando o desempenho no conjunto de validação parar de melhorar.
#
# SPLIT DE VALIDAÇÃO CRONOLÓGICO:
# Usamos os últimos 18 meses do período de treino como validação.
# Motivação: em séries temporais, NÃO se deve fazer split aleatório — isso causaria
# leakage (o modelo treinaria em dados futuros ao passado de validação).
# O split cronológico simula o cenário real: treinamos no passado e avaliamos no futuro.
#
# CALLBACKS (funções chamadas ao final de cada época):
#
# • EarlyStopping (patience=25):
#   Para o treinamento se o val_loss não melhorar em 25 épocas consecutivas.
#   restore_best_weights=True: restaura os pesos da época com menor val_loss,
#   descartando os pesos das épocas de piora (sem custo computacional extra).
#
# • ReduceLROnPlateau (patience=10, factor=0.5):
#   Se o val_loss não melhorar em 10 épocas, reduz a taxa de aprendizado pela
#   metade (ex.: 0.001 → 0.0005 → 0.00025). Isso permite 'refinar' a solução
#   em regiões planas do espaço de parâmetros, extraindo mais performance
#   sem risco de divergir com LR muito alta.
# =============================================================================

# Define o início da janela de validação: 18 meses antes do último jogo histórico.
# Jogos nesta janela são usados para monitorar val_loss, mas NÃO atualizam os pesos.
val_start = DATA_MAX - pd.DateOffset(months=18)
is_val = (train_df["date"] >= val_start).values  # máscara booleana

# Split cronológico: ~is_val para treino, is_val para validação
Xtr, Ytr, Wtr = Xs[~is_val], Y[~is_val], W[~is_val]  # dados de treino
Xva, Yva, Wva = Xs[is_val],  Y[is_val],  W[is_val]   # dados de validação

# Definição dos callbacks de controle de treinamento
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",         # métrica monitorada
        patience=25,                # número de épocas sem melhora antes de parar
        restore_best_weights=True   # restaura o melhor modelo encontrado
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,    # divide a LR pela metade quando plato detectado
        patience=10,   # épocas sem melhora para acionar a redução
        min_lr=1e-5    # limite inferior da taxa de aprendizado
    ),
]

# Treinamento:
# - epochs=200: teto máximo (early stopping geralmente para bem antes)
# - batch_size=256: mini-batch grande, adequado para o volume de dados (~19k amostras)
# - verbose=0: suprime a barra de progresso por época (saída limpa)
# - sample_weight=Wtr: aplica os pesos de decay temporal na loss
hist = model.fit(
    Xtr, Ytr, sample_weight=Wtr,
    validation_data=(Xva, Yva, Wva),
    epochs=200, batch_size=256,
    verbose=0, callbacks=callbacks
)
print(f"Melhor val_loss obtido: {min(hist.history['val_loss']):.4f}")

Melhor val_loss obtido: 0.5553


## 6. Avaliação e Inferência futura

In [30]:
# =============================================================================
# CÉLULA 11 — Conversão de Lambda para Placar e Avaliação no Conjunto de Validação
# =============================================================================
# OBJETIVO: (A) Definir como converter o λ contínuo em gols inteiros via moda
# da Poisson; (B) Avaliar o modelo no conjunto de validação com duas métricas
# interpretáveis: MAE de gols e acerto de resultado (1/X/2).
#
# CONVERSÃO LAMBDA → PLACAR (score_from_lambda):
# A rede prevê λ (gols esperados contínuos), mas o bolão exige placares inteiros.
# A conversão usa a MODA da distribuição de Poisson:
#   moda(Poisson(λ)) = floor(λ) para λ ≥ 1, e 0 para λ < 1
# Na prática calculamos a PMF (probabilidade de cada k) para k ∈ {0,1,2,3,4,5}
# e tomamos o argmax — o valor mais provável. O kmax=5 é o limite superior
# realista para Copa do Mundo (goleadas maiores são raríssimas).
#
# MÉTRICAS:
# - MAE (Mean Absolute Error) de gols: média da diferença absoluta entre gols
#   previstos e reais, por seleção. MAE = 0.978 significa erro médio de ~1 gol.
# - Acerto 1X2 (60.5%): percentual de jogos onde o modelo acerta o resultado
#   (vitória A / empate / vitória B). Benchmarks típicos: 45-55% para modelos
#   simples, 55-65% para modelos avançados — 60.5% é competitivo.
# =============================================================================

def score_from_lambda(lam, kmax=5):
    """
    Converte vetor de lambdas em placares inteiros via moda da Poisson.

    Args:
        lam: array de λ (gols esperados), shape (N,)
        kmax: número máximo de gols possível (default=5)

    Returns:
        Array de placares inteiros, shape (N,), valores em [0, kmax].
    """
    ks = np.arange(0, kmax + 1)  # possíveis valores de gols: [0, 1, 2, 3, 4, 5]
    # Calcula P(X=k | λ) para cada combinação de (jogo, k): shape (N, kmax+1)
    # np.clip(lam, 1e-6, None) evita λ=0 que causaria log(0) internamente
    pmf = poisson.pmf(ks[None, :], np.clip(lam, 1e-6, None)[:, None])
    return pmf.argmax(axis=1).astype(int)  # índice do k mais provável = moda

def outcome(gh, ga):
    """
    Converte placares em resultado 1/X/2:
    0 = vitória do mandante, 1 = empate, 2 = vitória do visitante.
    """
    return np.where(gh > ga, 0, np.where(gh == ga, 1, 2))

# Inferência no conjunto de validação (modo de avaliação: dropout desabilitado)
lam_va = model.predict(Xva, verbose=0)  # shape (N_val, 2)

# Converte lambdas em placares inteiros para mandante e visitante
gh_net = score_from_lambda(lam_va[:,0])  # gols previstos do mandante
ga_net = score_from_lambda(lam_va[:,1])  # gols previstos do visitante

# Baseline ingênuo: prevê sempre o placar médio do conjunto de treino.
# Serve como referência mínima — o modelo deve superar este baseline.
mh, ma = int(round(Ytr[:,0].mean())), int(round(Ytr[:,1].mean()))
gh_bl  = np.full(len(Xva), mh)  # baseline: sempre mesmo placar para mandante
ga_bl  = np.full(len(Xva), ma)  # baseline: sempre mesmo placar para visitante

# MAE: média do erro absoluto nos gols (mandante + visitante, dividido por 2)
mae_n = (np.abs(gh_net - Yva[:,0]) + np.abs(ga_net - Yva[:,1])).mean() / 2

# Acurácia 1X2: comparação do resultado previsto com o resultado real
acc_n = (outcome(gh_net, ga_net) == outcome(Yva[:,0], Yva[:,1])).mean()

print(f"Rede Neural c/ Poisson Calibrada | MAE Gols: {mae_n:.3f} | Acerto 1X2: {acc_n:5.1%}")

Rede Neural c/ Poisson Calibrada | MAE Gols: 0.978 | Acerto 1X2: 60.5%


In [31]:
# =============================================================================
# CÉLULA 12 — Construção das Features para Inferência (Copa 2026)
# =============================================================================
# OBJETIVO: Extrair as features de cada seleção participante dos 24 jogos com
# base no estado atual (Elo final, forma recente, ranking FIFA mais recente),
# aplicando encolhimento bayesiano para seleções com histórico esparso.
#
# ENCOLHIMENTO BAYESIANO POR CONFEDERAÇÃO (Bayesian Shrinkage / Transfer Learning):
# Seleções com poucos jogos recentes (ex.: Haiti, Curaçao) têm estimativas de
# forma e Elo muito incertas. Para reduzir essa incerteza, usamos o princípio
# do encolhimento: "puxamos" a estimativa da seleção em direção à média da
# sua confederação (CAF, CONCACAF, etc.), com peso inversamente proporcional
# ao número de jogos recentes.
#
# Fórmula de encolhimento (sh):
#   valor_ajustado = (n * valor_proprio + K * media_confederacao) / (n + K)
#   → Se n >> K: valor_proprio domina (time com bastante histórico)
#   → Se n << K: media_confederacao domina (time com histórico esparso)
#   K_SHRINK=8 significa que um time precisa de ao menos 8 jogos recentes para
#   sua estatística própria ter o mesmo peso que a média da confederação.
#
# FORMA RECENTE: janela de 6 anos (2020-2026) para calcular o número de jogos
# recentes — mais restrito que a janela de treino para evitar dados muito antigos.
# =============================================================================

last_form, last_rank, last_pts, n_recent = {}, {}, {}, {}

# Janela de 6 anos para contar jogos recentes (determina o peso no encolhimento)
recent_cut = DATA_MAX - pd.DateOffset(years=6)
recent_df = df[df["date"] >= recent_cut]

# Para cada seleção: extrai a forma atual (já no estado final após todos os jogos)
# e conta quantos jogos recentes ela tem no histórico.
for abrv, name in ABRV2NAME.items():
    last_form[abrv] = form_stats(name)  # tupla (gf_medio, ga_medio, ppg_medio)
    # Conta jogos como mandante OU visitante nos últimos 6 anos
    n_recent[abrv] = int(((recent_df["home_team"]==name)|(recent_df["away_team"]==name)).sum())

# Extrai ranking, pontos e confederação mais recentes de cada seleção.
# drop_duplicates(keep='last') após sort garante o registro mais atual.
fifa_latest = fifa.sort_values("rank_date").drop_duplicates("country_abrv", keep="last")
abrv2rank = fifa_latest.set_index("country_abrv")["rank"].to_dict()
abrv2pts  = fifa_latest.set_index("country_abrv")["total_points"].to_dict()
abrv2conf = fifa_latest.set_index("country_abrv")["confederation"].to_dict()

# Calcula médias de forma e Elo por confederação — usadas como âncora do encolhimento.
conf_form = defaultdict(list)   # confederação → lista de (gf, ga, ppg)
conf_elo  = defaultdict(list)   # confederação → lista de elo
for abrv, name in ABRV2NAME.items():
    c = abrv2conf.get(abrv, "OTHER")
    gf, ga, ppg = last_form[abrv]
    if not np.isnan(gf): conf_form[c].append((gf, ga, ppg))
    conf_elo[c].append(ELO_FINAL.get(name, ELO_BASE))

# Média global de forma: fallback para confederações com dados insuficientes
glob_form = np.nanmean([v for vs in conf_form.values() for v in vs], axis=0)

def conf_mean_form(c):
    """Retorna a média de forma da confederação c, ou a média global como fallback."""
    vs = conf_form.get(c, [])
    return np.mean(vs, axis=0) if vs else glob_form

K_SHRINK = 8.0  # força do prior de confederação: 8 jogos equivale à média da confederação

def team_features(abrv):
    """
    Extrai e retorna o dicionário de features de uma seleção para a inferência de 2026.
    Aplica encolhimento bayesiano nas estatísticas de forma para times com
    histórico esparso, usando a média da confederação como prior.

    Args:
        abrv: sigla FIFA de 3 letras da seleção

    Returns:
        Dicionário com: elo, rank, pts, gf (gols marcados médios), ga (gols sofridos
        médios), ppg (pontos por jogo médio) e n (número de jogos recentes).
    """
    name = ABRV2NAME[abrv]
    c = abrv2conf.get(abrv, "OTHER")       # confederação do time
    elo_t = ELO_FINAL.get(name, ELO_BASE)  # Elo final após todos os jogos históricos
    gf, ga, ppg = last_form[abrv]          # forma bruta
    cgf, cga, cppg = conf_mean_form(c)     # média da confederação (prior)
    n = n_recent[abrv]                     # número de jogos recentes

    # Se a seleção não tem forma calculada (sem histórico), usa diretamente a média da confederação
    if np.isnan(gf): gf, ga, ppg = cgf, cga, cppg; n = 0

    # Encolhimento bayesiano: sh(v, cv) = (n*v + K*cv) / (n + K)
    sh = lambda v, cv: (n * v + K_SHRINK * cv) / (n + K_SHRINK)
    gf, ga, ppg = sh(gf, cgf), sh(ga, cga), sh(ppg, cppg)

    rank = float(abrv2rank.get(abrv, DEF_RANK))
    pts  = float(abrv2pts.get(abrv, DEF_PTS))
    return dict(elo=elo_t, rank=rank, pts=pts, gf=gf, ga=ga, ppg=ppg, n=n_recent[abrv])

In [32]:
# =============================================================================
# CÉLULA 13 — Inferência Final: Previsão dos 24 Placares da Copa 2026
# =============================================================================
# OBJETIVO: Montar o vetor de features de cada jogo (mesma estrutura usada no
# treino), normalizar com o scaler ajustado, rodar a rede neural treinada e
# converter os lambdas previstos em placares inteiros determinísticos.
#
# FLUXO DE INFERÊNCIA:
# 1. Para cada jogo i dos 24 fixtures:
#    a. Extrai features de A e B via team_features() (Elo, ranking, forma)
#    b. Adiciona a feature 'neutral' (0=sede, 1=campo neutro)
#    c. Adiciona as odds reais convertidas (imp_1, imp_x, imp_2) da célula 3
#    d. Monta o vetor de 17 features na MESMA ORDEM que FEATS
# 2. Aplica scaler.transform() com os parâmetros do treino
# 3. model.predict() → (λ_A, λ_B) para cada jogo
# 4. score_from_lambda() → placar inteiro via moda da Poisson
#
# CONSISTÊNCIA COM O TREINO:
# É crucial que a ordem de features em build_match_row() seja IDÊNTICA à lista
# FEATS — qualquer desalinhamento causaria previsões sem sentido (features trocadas).
# =============================================================================

def build_match_row(r):
    """
    Monta o vetor de features de um confronto para inferência.

    Args:
        r: linha do DataFrame cal com metadados do jogo (A, B, A_sede, imp_*)

    Returns:
        Lista de 17 floats na mesma ordem da lista FEATS:
        [elo_h, elo_a, elo_diff, rank_h, rank_a, pts_h, pts_a,
         f_gf_h, f_ga_h, f_ppg_h, f_gf_a, f_ga_a, f_ppg_a,
         neutral, imp_1, imp_x, imp_2]
    """
    A = team_features(r["A"])  # features da seleção A
    B = team_features(r["B"])  # features da seleção B

    # neutral: 0 se A é país-sede (vantagem de campo), 1 se campo neutro
    neutral = 0 if r["A_sede"] else 1

    # Retorna vetor na ordem exata de FEATS (crítico para consistência com o treino)
    return [
        A["elo"], B["elo"], A["elo"]-B["elo"],  # Elo absoluto e diferencial
        A["rank"], B["rank"],                    # ranking FIFA
        A["pts"], B["pts"],                      # pontos FIFA
        A["gf"], A["ga"], A["ppg"],              # forma de A
        B["gf"], B["ga"], B["ppg"],              # forma de B
        neutral,                                  # mando de campo
        r["imp_1"], r["imp_x"], r["imp_2"]      # probabilidades das odds reais de 2026
    ]

# Monta a matriz de features para os 24 jogos: shape (24, 17)
Xfx = np.array([build_match_row(r) for _, r in cal.iterrows()], dtype="float32")

# Normaliza com o scaler ajustado no treino — usa os MESMOS μ e σ
# (não ajusta de novo nos dados de 2026, o que seria leakage)
Xfx_s = scaler.transform(Xfx).astype("float32")

# Inferência: modelo retorna (λ_A, λ_B) para cada um dos 24 jogos
# verbose=0 suprime a barra de progresso (inútil para apenas 24 amostras)
lam_fx = model.predict(Xfx_s, verbose=0)  # shape (24, 2)

# Salva os lambdas como colunas intermediárias (útil para debug e análise)
cal["lam_A"] = lam_fx[:,0]  # gols esperados de A (contínuo)
cal["lam_B"] = lam_fx[:,1]  # gols esperados de B (contínuo)

# Converte lambdas em placares inteiros via moda da Poisson
cal["gols_A"] = score_from_lambda(lam_fx[:,0])  # placar final de A
cal["gols_B"] = score_from_lambda(lam_fx[:,1])  # placar final de B

## 7. Exportação e Validação Final do JSON

In [33]:
# =============================================================================
# CÉLULA 14 — Exportação dos Resultados em JSON (Formato do Bolão)
# =============================================================================
# OBJETIVO: Serializar as previsões dos 24 placares no formato JSON exato
# exigido pelo arquivo bolao_copa.txt e salvar em disco.
#
# ESTRUTURA DO JSON GERADO:
# {
#   "nome": "Bruno Aires",
#   "turma": "2º BIM 2026",
#   "resultados": {
#     "jogo1": { "MEX": {"gols": 2}, "RSA": {"gols": 0} },
#     "jogo2": { "KOR": {"gols": 1}, "CZE": {"gols": 1} },
#     ...
#     "jogo24": { "UZB": {"gols": 0}, "COL": {"gols": 2} }
#   }
# }
#
# DETALHES TÉCNICOS:
# - int(r["gols_A"]): conversão explícita de numpy.int64 para int Python.
#   json.dump() não serializa numpy types nativamente — causaria TypeError.
# - ensure_ascii=False: preserva caracteres especiais (ex.: acentos em nomes)
# - indent=2: formata o JSON com indentação de 2 espaços (legibilidade humana)
# - sort_values("jogo"): garante que os jogos aparecem em ordem numérica no JSON
# =============================================================================

NOME, TURMA = "Bruno Aires", "2º BIM 2026"  # identificação do aluno no arquivo de saída

# Constrói o dicionário de resultados iterando sobre os jogos em ordem.
# Chave: "jogo{n}" (string com número), Valor: dicionário com gols de cada seleção.
resultados = {}
for _, r in cal.sort_values("jogo").iterrows():
    resultados[f"jogo{int(r['jogo'])}"] = {
        r["A"]: {"gols": int(r["gols_A"])},  # int() converte numpy.int64 → Python int
        r["B"]: {"gols": int(r["gols_B"])},
    }

# Monta o dicionário raiz com metadados + resultados
saida = {"nome": NOME, "turma": TURMA, "resultados": resultados}

# Serializa para JSON e salva no arquivo de saída
OUT = "bolao_resultado.json"
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(saida, f, ensure_ascii=False, indent=2)

print("JSON salvo com sucesso em:", OUT)

JSON salvo com sucesso em: bolao_resultado.json
